# 03 – Modellvergleich, Hyperparameteroptimierung & Ergebnisse
## ADSC21 Prüfungsleistung – Milwaukee Property Sales

**Hauptnotebook:** Vergleich von Decision Tree, Random Forest und Gradient Boosting gegenüber der Baseline (Lineare Regression). Hyperparameteroptimierung mit RandomizedSearchCV, Kreuzvalidierung, separater Test und KPI-Darstellung.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint, uniform

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
print('Libraries loaded.')

## 1. Daten laden & Split

In [ ]:
df = pd.read_csv('../data/property_sales_clean.csv')

price_col = 'sale_price' if 'sale_price' in df.columns else 'saleprice'
numerical_features = [c for c in ['fin_sqft', 'year_built', 'bedrooms', 'bathrooms', 'lotsize', 'stories']
                      if c in df.columns]
categorical_features = [c for c in ['style', 'extwall', 'zip_code'] if c in df.columns]

X = df[numerical_features + categorical_features]
y = df[price_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. Preprocessor definieren

In [ ]:
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=10))
])
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
], remainder='drop')

print('Preprocessor bereit.')

## 3. Modelle definieren & mit Cross-Validation vergleichen

5-Fold Cross-Validation auf dem Trainings-Set. Metrik: negMSE → RMSE.

In [ ]:
models = {
    'Lineare Regression (Baseline)': LinearRegression(),
    'Decision Tree':                 DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
    'Random Forest':                 RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting':             GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                                               max_depth=4, random_state=RANDOM_STATE)
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X_train, y_train,
                             cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    rmse_cv = np.sqrt(-scores)
    results[name] = {'rmse_cv_mean': rmse_cv.mean(), 'rmse_cv_std': rmse_cv.std()}
    print(f'{name:35s} → CV-RMSE: ${rmse_cv.mean():,.0f} ± ${rmse_cv.std():,.0f}')

## 4. Hyperparameteroptimierung – Random Forest

RandomizedSearchCV durchsucht einen großen Parameterraum effizienter als GridSearchCV
(vgl. Géron 2022, Kap. 2). Wir suchen 30 Kombinationen über 3-Fold-CV.

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

param_distributions = {
    'model__n_estimators':     randint(100, 500),
    'model__max_depth':        [None, 10, 15, 20, 30],
    'model__min_samples_split': randint(2, 20),
    'model__min_samples_leaf':  randint(1, 10),
    'model__max_features':     ['sqrt', 'log2', 0.5]
}

rf_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='neg_mean_squared_error',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

best_params = rf_search.best_params_
best_cv_rmse = np.sqrt(-rf_search.best_score_)
print(f'\nBeste Parameter: {best_params}')
print(f'Bester CV-RMSE:  ${best_cv_rmse:,.0f}')

## 5. Finales Modell auf Test-Set evaluieren

In [ ]:
best_rf = rf_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)

# Alle Modelle auf Test-Set auswerten
def evaluate(y_true, y_pred, name=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    # MdAPE als KPI (robust gegen Ausreißer)
    mdape = np.median(np.abs((y_true - y_pred) / y_true)) * 100
    return {'Modell': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MdAPE(%)': mdape}

test_results = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    test_results.append(evaluate(y_test, y_pred, name))

# Bestes RF (nach Tuning)
test_results.append(evaluate(y_test, y_pred_rf, 'Random Forest (tuned)'))

df_results = pd.DataFrame(test_results).set_index('Modell')
df_results = df_results.sort_values('RMSE')
print(df_results.to_string(float_format=lambda x: f'{x:,.1f}'))
df_results.to_csv('../data/model_results.csv')

## 6. Visualisierung der Ergebnisse
### 6.1 Modellvergleich – RMSE

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if 'Baseline' in m else '#1f77b4' if 'tuned' not in m else '#2ca02c'
          for m in df_results.index]
bars = ax.barh(df_results.index[::-1], df_results['RMSE'][::-1] / 1000, color=colors[::-1])
ax.set_xlabel('RMSE (Tsd. USD)')
ax.set_title('Modellvergleich: Test-RMSE', fontsize=13)
for bar, val in zip(bars, df_results['RMSE'][::-1] / 1000):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'${val:.1f}k', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../report/abb7_modellvergleich_rmse.png', bbox_inches='tight')
plt.show()

### 6.2 Predicted vs. Actual – bestes Modell

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_test / 1000, y_pred_rf / 1000, alpha=0.3, s=10, color='steelblue')
lim = [0, max(y_test.max(), y_pred_rf.max()) / 1000]
ax.plot(lim, lim, 'r--', lw=1.5, label='Perfekte Vorhersage')
ax.set_xlabel('Tatsächlicher Preis (Tsd. USD)')
ax.set_ylabel('Vorhergesagter Preis (Tsd. USD)')
ax.set_title('Predicted vs. Actual – Random Forest (tuned)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('../report/abb8_predicted_vs_actual_rf.png', bbox_inches='tight')
plt.show()

### 6.3 Feature Importance (Top 15)

In [ ]:
rf_model = best_rf.named_steps['model']
prep     = best_rf.named_steps['preprocessor']

# Feature-Namen aus Pipeline extrahieren
try:
    cat_names = prep.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_features).tolist()
except AttributeError:
    cat_names = [f'cat_{i}' for i in range(rf_model.n_features_in_ - len(numerical_features))]

feature_names = numerical_features + cat_names
importances   = rf_model.feature_importances_

fi = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
fi[::-1].plot.barh(ax=ax, color='steelblue')
ax.set_title('Feature Importance – Random Forest (Top 15)', fontsize=12)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../report/abb9_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top-5 Features:')
print(fi.head())

### 6.4 KPI-Dashboard: Vorhersagen verdichtet

Als KPI nutzen wir den **Median Absolute Percentage Error (MdAPE)** und die **Trefferquote ±15%**
(Anteil Vorhersagen innerhalb von 15% des tatsächlichen Preises).

In [ ]:
residuals = y_test - y_pred_rf
pct_error = np.abs(residuals / y_test) * 100

within_10 = (pct_error <= 10).mean() * 100
within_15 = (pct_error <= 15).mean() * 100
within_20 = (pct_error <= 20).mean() * 100
mdape     = np.median(pct_error)

print('=== KPI – Random Forest (tuned) ===')
print(f'MdAPE:              {mdape:.1f}%')
print(f'Trefferquote ±10%:  {within_10:.1f}%')
print(f'Trefferquote ±15%:  {within_15:.1f}%')
print(f'Trefferquote ±20%:  {within_20:.1f}%')

# Residuenverteilung
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(residuals / 1000, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', lw=1.5, ls='--')
axes[0].set_title('Residuenverteilung (Tsd. USD)', fontsize=11)
axes[0].set_xlabel('Residuum (Tsd. USD)')

axes[1].hist(pct_error.clip(0, 50), bins=50, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].axvline(mdape, color='red', lw=1.5, ls='--', label=f'MdAPE: {mdape:.1f}%')
axes[1].set_title('Prozentualer absoluter Fehler (Verteilung)', fontsize=11)
axes[1].set_xlabel('APE (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../report/abb10_kpi_residuen.png', bbox_inches='tight')
plt.show()

## 7. Bestes Modell speichern

In [ ]:
joblib.dump(best_rf, '../data/best_model_rf_tuned.pkl')
print('Modell gespeichert: data/best_model_rf_tuned.pkl')

# Ergebnisse speichern
final_metrics = {
    'best_model': 'Random Forest (tuned)',
    'test_rmse': float(np.sqrt(mean_squared_error(y_test, y_pred_rf))),
    'test_mae':  float(mean_absolute_error(y_test, y_pred_rf)),
    'test_r2':   float(r2_score(y_test, y_pred_rf)),
    'test_mdape': float(mdape),
    'within_15pct': float(within_15),
    'best_params': {k.replace('model__',''): v for k, v in best_params.items()}
}
with open('../data/final_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

print(json.dumps(final_metrics, indent=2))